# Question 1 — Qu'est-ce qui fait vraiment le remplissage ?

## 1. Démarche

Question d'**explication** : quelles caractéristiques d'un événement (prix, notoriété, genre, salle, jour) sont associées au taux de remplissage, **toutes choses égales par ailleurs** ?

L'outil adapté parmi les méthodes du cours est la **régression linéaire multiple**, dont les coefficients se lisent directement. Nous la complétons par :
- une **validation croisée** pour vérifier que le modèle prédit mieux qu'une simple moyenne ;
- une comparaison avec un **KNN** (régression par plus proches voisins) : si le KNN ne fait pas mieux, une relation linéaire suffit.

K-means et DBSCAN sont des méthodes non supervisées (groupes, anomalies) : elles ne répondent pas à une question d'explication d'une variable cible, nous ne les utilisons pas ici.

## 2. Chargement des données

On travaille sur le fichier nettoyé des événements. Les 3 événements dont les ventes sont inconnues (EV374, EV303, EV355) sont exclus de Q1 à Q4, conformément à `preprocessing.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

evenements = pd.read_csv(
    '../data/processed/evenements_clean.csv',
    parse_dates=['date']
)

# Q1 : on garde uniquement les événements dont les ventes sont connues
data = evenements.dropna(subset=['billets_vendus', 'taux_remplissage']).copy()

# billets_vendus est de type Int64 (nullable) : conversion en float
data['billets_vendus'] = data['billets_vendus'].astype(float)

# La capacité correspond à 3 jauges fixes : on la traite comme catégorielle
data['capacite_cat'] = data['capacite'].astype(int).astype(str)

# Événements complets : ventes plafonnées par la jauge (effet plafond)
data['complet'] = (data['taux_remplissage'] >= 1).astype(int)

print("Nombre d'événements analysés :", len(data), "(attendu : 96 - 3 = 93)")
print("Événements complets :", data['complet'].sum(), "(attendu : 5)")
assert '6500' in data['capacite_cat'].values, "la capacité de référence 6500 est absente"
display(data.head())

## 3. Compréhension du taux de remplissage

Le taux de remplissage est la variable cible : `billets_vendus / capacite`.

In [ ]:
print("Statistiques du taux de remplissage")
display(data['taux_remplissage'].describe().round(3))

plt.figure(figsize=(8, 5))
sns.histplot(data['taux_remplissage'], bins=12, kde=True)
plt.title("Distribution du taux de remplissage")
plt.xlabel("Taux de remplissage")
plt.ylabel("Nombre d'événements")
plt.show()

## 4. Variables numériques : prix et notoriété

On examine graphiquement les relations entre le prix, la notoriété et le taux de remplissage. Le prix et la notoriété peuvent être liés entre eux : un artiste très connu fait payer plus cher. C'est pourquoi une corrélation simple ne suffit pas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.regplot(data=data, x='prix_moyen_eur', y='taux_remplissage',
            scatter_kws={'alpha': 0.7}, ax=axes[0])
axes[0].set_title("Prix moyen et taux de remplissage")
axes[0].set_xlabel("Prix moyen (€)")
axes[0].set_ylabel("Taux de remplissage")
sns.regplot(data=data, x='notoriete_artiste', y='taux_remplissage',
            scatter_kws={'alpha': 0.7}, ax=axes[1])
axes[1].set_title("Notoriété de l'artiste et taux de remplissage")
axes[1].set_xlabel("Notoriété de l'artiste")
axes[1].set_ylabel("Taux de remplissage")
plt.tight_layout()
plt.show()

## 5. Variables catégorielles : genre, jour, capacité

On compare les moyennes du taux de remplissage par modalité. Les effectifs par modalité sont faibles : ces comparaisons sont des tendances, que le modèle multivarié précise ensuite.

`jour_semaine` est retenu à la place de `date` : les deux colonnes se contredisent dans 81 cas sur 98 (niveau du hasard), et `jour_semaine` a un profil plus réaliste (samedi dominant, aucun lundi). Ce choix repose sur la plausibilité et reste signalé comme une limite.

La capacité prend 3 valeurs (6 500, 9 800, 15 800 places) : c'est une configuration de salle, pas une variable continue.

In [ ]:
for var, ordre in [('genre', None), ('jour_semaine', None), ('capacite_cat', ['6500', '9800', '15800'])]:
    tab = (data.groupby(var)['taux_remplissage']
               .agg(['count', 'mean', 'median', 'std'])
               .sort_values('mean', ascending=False))
    if var == 'capacite_cat':
        tab['complets'] = data.groupby(var)['complet'].sum()
    print(f"--- {var} ---")
    display(tab.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.boxplot(data=data, x='genre', y='taux_remplissage', ax=axes[0])
axes[0].set_title("Selon le genre")
axes[0].tick_params(axis='x', rotation=40)
sns.boxplot(data=data, x='jour_semaine', y='taux_remplissage', ax=axes[1])
axes[1].set_title("Selon le jour")
axes[1].tick_params(axis='x', rotation=40)
sns.boxplot(data=data, x='capacite_cat', y='taux_remplissage',
            order=['6500', '9800', '15800'], ax=axes[2])
axes[2].set_title("Selon la capacité (places)")
for ax in axes:
    ax.set_ylabel("Taux de remplissage")
plt.tight_layout()
plt.show()

## 6. Modèle de régression linéaire multiple

On mesure l'association de chaque facteur avec le remplissage **toutes choses égales par ailleurs**.

Les modalités de référence sont fixées explicitement (la plus fréquente pour le genre et le jour, 6 500 places pour la capacité) pour que les coefficients soient lisibles. Sans cela, `statsmodels` prend l'ordre alphabétique, et `'15800'` passerait en référence pour la capacité. Un coefficient de modalité se lit comme **l'écart de taux de remplissage par rapport à la référence**.

In [ ]:
CIBLE = 'taux_remplissage'

ref_genre = data['genre'].value_counts().idxmax()
ref_jour = data['jour_semaine'].value_counts().idxmax()
print("Références -> genre :", ref_genre, "| jour :", ref_jour, "| capacité : 6500")

TERMES = {
    'prix':      'prix_moyen_eur',
    'notoriete': 'notoriete_artiste',
    'genre':     f'C(genre, Treatment(reference="{ref_genre}"))',
    'capacite':  'C(capacite_cat, Treatment(reference="6500"))',
    'jour':      f'C(jour_semaine, Treatment(reference="{ref_jour}"))',
}

def formule(cible, termes):
    return cible + ' ~ ' + ' + '.join(TERMES[t] for t in termes)

FORMULE_Q1 = formule(CIBLE, TERMES.keys())
modele_q1 = smf.ols(FORMULE_Q1, data=data).fit()

print(f"Observations : {int(modele_q1.nobs)} | paramètres : {len(modele_q1.params)}")
print(modele_q1.summary())

## 7. Coefficients et qualité du modèle

In [ ]:
ic = modele_q1.conf_int()
coefficients = pd.DataFrame({
    'coefficient': modele_q1.params,
    'IC95_bas': ic[0],
    'IC95_haut': ic[1],
    'p_value': modele_q1.pvalues
})
display(coefficients.round(4))

print(f"R² = {modele_q1.rsquared:.3f} | R² ajusté = {modele_q1.rsquared_adj:.3f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(x=modele_q1.fittedvalues, y=modele_q1.resid)
plt.axhline(0, linestyle='--')
plt.title("Résidus du modèle Q1")
plt.xlabel("Valeurs prédites")
plt.ylabel("Résidus")
plt.show()

Lecture du graphique des résidus : ils doivent être répartis autour de 0 sans forme nette. Un effet « entonnoir » ou une courbure signalerait un modèle mal adapté. Les points les plus hauts des valeurs prédites correspondent aux événements proches de 100 %, plafonnés (voir section 10).

## 8. Importance relative des facteurs

Les p-values par modalité ne disent pas quel facteur compte le plus. On retire donc chaque facteur à tour de rôle et on mesure la **perte de R² ajusté** : plus elle est grande, plus le facteur apporte d'information. On donne aussi l'effet d'un écart-type des variables numériques, pour les comparer entre elles.

In [ ]:
lignes = []
for t in TERMES:
    reduit = smf.ols(formule(CIBLE, [k for k in TERMES if k != t]), data=data).fit()
    lignes.append({'facteur': t,
                   'perte_R2_ajuste': modele_q1.rsquared_adj - reduit.rsquared_adj})

importance = pd.DataFrame(lignes).sort_values('perte_R2_ajuste', ascending=False)
display(importance.round(4))

for var in ['prix_moyen_eur', 'notoriete_artiste']:
    effet = modele_q1.params[var] * data[var].std()
    print(f"{var} : +1 écart-type ({data[var].std():.1f}) -> {effet:+.3f} de taux de remplissage")

## 9. Validation du modèle : régression linéaire et KNN

Le R² du modèle précédent est calculé sur les mêmes données que l'estimation : il est optimiste. Avec 93 événements, on utilise une **validation croisée à 5 plis** : le modèle est estimé sur 4/5 des événements et testé sur le 1/5 restant, 5 fois.

On compare :
- une **référence** qui prédit toujours la moyenne (R² ≈ 0 : le minimum à battre) ;
- la **régression linéaire complète** ;
- la régression linéaire **sans le prix** : si le R² ne baisse pas, le prix n'apporte rien à la prédiction ;
- un **KNN** (5 voisins) : si le KNN ne fait pas mieux que la régression linéaire, l'hypothèse de linéarité est suffisante.

In [ ]:
NUM = ['prix_moyen_eur', 'notoriete_artiste']
CAT = ['genre', 'capacite_cat', 'jour_semaine']

def pipeline(modele, num=NUM, cat=CAT):
    """Standardise les variables numériques, encode les catégorielles (one-hot), puis applique le modèle."""
    prep = ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])
    return make_pipeline(prep, modele)

CV = KFold(n_splits=5, shuffle=True, random_state=42)   # petit échantillon : validation croisée

def evaluer(nom, modele, X, y):
    res = cross_validate(modele, X, y, cv=CV,
                         scoring=['r2', 'neg_mean_absolute_error'])
    return {'modèle': nom,
            'R² (CV 5 plis)': res['test_r2'].mean(),
            'écart-type R²': res['test_r2'].std(),
            'MAE (CV)': -res['test_neg_mean_absolute_error'].mean()}

X = data[NUM + CAT]
y = data[CIBLE]

resultats = pd.DataFrame([
    evaluer('Régression linéaire complète', pipeline(LinearRegression()), X, y),
    evaluer('Régression linéaire sans prix', pipeline(LinearRegression(), num=['notoriete_artiste']), X, y),
    evaluer('KNN (k = 5)',                pipeline(KNeighborsRegressor(n_neighbors=5)), X, y),
])
display(resultats.round(3))

# Contrôle sur un découpage apprentissage / test classique (80 % / 20 %)
X_app, X_test, y_app, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg = pipeline(LinearRegression()).fit(X_app, y_app)
pred = reg.predict(X_test)
print(f"Découpage 80/20 -> R² test = {r2_score(y_test, pred):.3f} | "
      f"MAE test = {mean_absolute_error(y_test, pred):.3f} "
      f"(soit {mean_absolute_error(y_test, pred)*100:.1f} points de remplissage)")

## 10. Robustesse : l'effet plafond des événements complets

Le taux est borné à 100 % et 5 événements sont complets (dont 4 en 6 500 places) : pour eux, le taux ne mesure pas toute la demande, et la régression les traite comme des valeurs ordinaires. On réestime le modèle **sans eux** et on compare les coefficients du prix et de la notoriété : s'ils sont proches, le plafond ne pilote pas les conclusions.

In [ ]:
m_sans_complets = smf.ols(FORMULE_Q1, data=data[data['complet'] == 0]).fit()

lignes = []
for var in ['prix_moyen_eur', 'notoriete_artiste']:
    for nom, m in [('Avec complets', modele_q1), ('Sans complets', m_sans_complets)]:
        ic_m = m.conf_int().loc[var]
        lignes.append({'variable': var, 'modèle': nom,
                       'coefficient': m.params[var],
                       'IC95_bas': ic_m[0], 'IC95_haut': ic_m[1],
                       'p_value': m.pvalues[var]})
display(pd.DataFrame(lignes).round(4))
print("Observations sans complets :", int(m_sans_complets.nobs))

## 11. Conclusion

La cellule suivante résume les résultats chiffrés, pour que la conclusion rédigée dans le rapport reprenne les valeurs réelles.

In [ ]:
signif = coefficients[(coefficients['p_value'] < 0.05) & (coefficients.index != 'Intercept')]
print("Coefficients significatifs à 5 % :")
display(signif.round(4) if len(signif) else "Aucun")

print("\nFacteurs classés par variance expliquée apportée :")
for _, r in importance.iterrows():
    print(f"  - {r['facteur']:<10} perte de R² ajusté = {r['perte_R2_ajuste']:.3f}")

lin = resultats.set_index('modèle').loc['Régression linéaire complète', 'R² (CV 5 plis)']
sans_prix = resultats.set_index('modèle').loc['Régression linéaire sans prix', 'R² (CV 5 plis)']
knn = resultats.set_index('modèle').loc['KNN (k = 5)', 'R² (CV 5 plis)']
print(f"\nR² en validation croisée : linéaire {lin:.2f} | sans prix {sans_prix:.2f} | KNN {knn:.2f}")
print("Le prix améliore-t-il la prédiction ?", "oui" if lin - sans_prix > 0.02 else "non (gain < 0,02 de R²)")
print("Le KNN fait-il mieux que la régression linéaire ?", "oui" if knn - lin > 0.02 else "non")

**Recommandation.** Agir sur les facteurs qui ressortent à la fois comme significatifs dans le modèle, importants dans la section 8 et stables sans les événements complets (section 10), plutôt que sur le prix seul. Un facteur significatif dans un seul de ces contrôles est une piste, pas un levier établi.

**Limites.**
- L'analyse est **observationnelle** : une association ne prouve pas qu'agir sur un facteur provoquerait la variation observée.
- Les **événements complets sont plafonnés à 100 %** : une partie de la demande réelle est masquée ; la section 10 mesure l'effet de ce plafond sans le supprimer.
- L'échantillon est **petit** (environ 93 événements pour une quinzaine de paramètres) : les coefficients par modalité sont peu précis, d'où la validation croisée.
- Le **jour de la semaine** est retenu sur un critère de plausibilité (incohérent avec la date dans 81 cas sur 98).
- Le **prix est choisi par les organisateurs** selon l'artiste et la salle : son coefficient n'est pas un effet causal.